# Experiment 10: Fine-Tuning for Domain Adaptation

## Objective

To fine-tune a language model on domain-specific data
and evaluate whether the model learns the specialized domain.

## Workflow

Domain Dataset
      ↓
Pre-trained Language Model
      ↓
Fine-Tuning
      ↓
Specialized Model
      ↓
Evaluation
      ↓
Compare Results

In [1]:
print("Fine-Tuning for Domain Adaptation")
print("Experiment 10")

Fine-Tuning for Domain Adaptation
Experiment 10


In [2]:
!pip -q install transformers datasets peft accelerate

In [3]:
domain_data = [
    {
        "question": "What is hypertension?",
        "answer": "Hypertension is a medical condition in which blood pressure remains higher than the normal range."
    },
    {
        "question": "What is diabetes?",
        "answer": "Diabetes is a chronic condition that affects how the body regulates blood glucose."
    },
    {
        "question": "What is a symptom?",
        "answer": "A symptom is a change or problem experienced by a person that may indicate a health condition."
    },
    {
        "question": "What is a diagnosis?",
        "answer": "A diagnosis is the identification of a disease or health condition based on clinical information and tests."
    },
    {
        "question": "What is preventive healthcare?",
        "answer": "Preventive healthcare focuses on reducing health risks and detecting diseases early."
    },
    {
        "question": "What is medical imaging?",
        "answer": "Medical imaging uses techniques such as X-rays, CT scans, and MRI to create images of structures inside the body."
    },
    {
        "question": "What is patient monitoring?",
        "answer": "Patient monitoring involves regularly observing and recording a patient's health measurements and condition."
    },
    {
        "question": "What is personalized medicine?",
        "answer": "Personalized medicine uses patient-specific information to help guide healthcare decisions and treatment."
    }
]

print("Number of training examples:", len(domain_data))

Number of training examples: 8


In [4]:
from datasets import Dataset

dataset = Dataset.from_list(domain_data)

print(dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 8
})


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Pre-trained model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Pre-trained model loaded successfully!


In [6]:
def format_example(example):
    return {
        "text": f"Question: {example['question']}\nAnswer: {example['answer']}"
    }

formatted_dataset = dataset.map(format_example)

print(formatted_dataset[0])

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

{'question': 'What is hypertension?', 'answer': 'Hypertension is a medical condition in which blood pressure remains higher than the normal range.', 'text': 'Question: What is hypertension?\nAnswer: Hypertension is a medical condition in which blood pressure remains higher than the normal range.'}


In [8]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=False
)

print(tokenized_dataset)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 8
})


In [9]:
split_dataset = tokenized_dataset.train_test_split(
    test_size=0.25,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Evaluation examples:", len(eval_dataset))

Training examples: 6
Evaluation examples: 2


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./healthcare_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

print("Training configuration created successfully!")

Training configuration created successfully!


In [14]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

print("Trainer created successfully!")

Trainer created successfully!


In [15]:
print("Starting fine-tuning...")

trainer.train()

print("Fine-tuning completed successfully!")

Starting fine-tuning...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,37.801327
2,33.452148
3,35.384789
4,33.760445
5,32.636261
6,30.819925
7,30.879959
8,30.408043
9,30.962114


Fine-tuning completed successfully!


In [16]:
eval_results = trainer.evaluate()

print("Evaluation Results:")
print(eval_results)

Training Loss,Validation Loss,Step
30.962114,33.005363,9


Evaluation Results:
{'eval_loss': 33.00536346435547}


In [17]:
test_questions = [
    "What is preventive healthcare?",
    "What is medical imaging?",
    "What is patient monitoring?"
]

for question in test_questions:
    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=50
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("Question:", question)
    print("Model Answer:", answer)
    print("-" * 60)

Question: What is preventive healthcare?
Model Answer: preventive medicine
------------------------------------------------------------
Question: What is medical imaging?
Model Answer: a vascular device
------------------------------------------------------------
Question: What is patient monitoring?
Model Answer: a syringe
------------------------------------------------------------


In [18]:
question = "What is hypertension?"
prompt = f"Question: {question}\nAnswer:"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

fine_tuned_answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Question:", question)
print("\nFine-Tuned Model Answer:")
print(fine_tuned_answer)

Question: What is hypertension?

Fine-Tuned Model Answer:
a symptom of a weakened immune system


In [19]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

original_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
original_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

print("Original pre-trained model loaded successfully!")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Original pre-trained model loaded successfully!


In [20]:
question = "What is hypertension?"
prompt = f"Question: {question}\nAnswer:"

# Original model
original_inputs = original_tokenizer(
    prompt,
    return_tensors="pt"
)

original_outputs = original_model.generate(
    **original_inputs,
    max_new_tokens=50
)

original_answer = original_tokenizer.decode(
    original_outputs[0],
    skip_special_tokens=True
)

# Fine-tuned model
fine_tuned_inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

fine_tuned_outputs = model.generate(
    **fine_tuned_inputs,
    max_new_tokens=50
)

fine_tuned_answer = tokenizer.decode(
    fine_tuned_outputs[0],
    skip_special_tokens=True
)

print("Question:", question)

print("\n--- Original Model ---")
print(original_answer)

print("\n--- Fine-Tuned Model ---")
print(fine_tuned_answer)

Question: What is hypertension?

--- Original Model ---
hypertension

--- Fine-Tuned Model ---
a symptom of a weakened immune system


# Experiment 10: Fine-Tuning for Domain Adaptation

## Objective

To fine-tune a pre-trained language model using domain-specific
healthcare data and evaluate its behavior after fine-tuning.

## Dataset

Domain: Healthcare

Training examples: 6
Evaluation examples: 2

## Model

Pre-trained model: google/flan-t5-small

## Fine-Tuning

The model was fine-tuned using the healthcare question-answer
dataset for 3 training epochs.

## Evaluation

The fine-tuned model was evaluated using a separate evaluation
dataset and tested on new healthcare questions.

## Before vs After

The original pre-trained model and the fine-tuned model were
tested using the same healthcare questions.

## Conclusion

Fine-tuning adapts a pre-trained language model to a specific
domain by training it on domain-specific examples. In this
experiment, healthcare question-answer data was used to adapt
the model toward healthcare-related responses.